# GLES Data Inspection

Goal: understand the raw `.sav` metadata file and the open-ended response CSV before creating a pilot sample.

## 1. Load packages and paths

The notebook lives in `notebooks/stance_ambiguity/`, so `../..` points back to the project root.

In [ ]:
from pathlib import Path
import re

import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]

RAW_DIR = PROJECT_ROOT / "data" / "stance_ambiguity" / "raw"

sav_path = RAW_DIR / "ZA10101_v3-0-0_en.sav"
open_ended_path = RAW_DIR / "ZA10101_v3-0-0_open-ended.csv"

print("Project root:", PROJECT_ROOT)
print("SAV exists:", sav_path.exists(), sav_path)
print("Open-ended CSV exists:", open_ended_path.exists(), open_ended_path)

Project root: /Users/jiayunjin/Documents/Research_codes/OpenCodeBook
SAV exists: True /Users/jiayunjin/Documents/Research_codes/OpenCodeBook/data/stance_ambiguity/raw/ZA10101_v3-0-0_en.sav
Open-ended CSV exists: True /Users/jiayunjin/Documents/Research_codes/OpenCodeBook/data/stance_ambiguity/raw/ZA10101_v3-0-0_open-ended.csv


## 2. Read the two raw files

Important: the open-ended CSV is semicolon-delimited and uses a Latin encoding. If read as UTF-8, German umlauts can break.

In [ ]:
sav = pd.read_spss(sav_path)
open_ended = pd.read_csv(open_ended_path, sep=";", encoding="latin1", dtype=str)

In [ ]:
def normalize_lfdn(series):
    return pd.to_numeric(series, errors="coerce").astype("Int64").astype(str)

sav["lfdn_key"] = normalize_lfdn(sav["lfdn"])
open_ended["lfdn_key"] = normalize_lfdn(open_ended["lfdn"])

print("SAV shape:", sav.shape)
print("Open-ended CSV shape:", open_ended.shape)

SAV shape: (8562, 804)
Open-ended CSV shape: (8562, 22)


## 3. What does one row represent?

`lfdn` is the respondent ID. If it is unique, then one row is one respondent, not one text answer.

In [28]:
print("Open-ended rows:", len(open_ended))
print("Unique lfdn values:", open_ended["lfdn"].nunique())
print("Duplicate lfdn rows:", open_ended.duplicated("lfdn").sum())

print("\nSAV rows:", len(sav))
print("SAV unique lfdn values:", sav["lfdn_key"].nunique())
print("lfdn values in open-ended but not in SAV:", (~open_ended["lfdn_key"].isin(sav["lfdn_key"])).sum())
print("lfdn values in SAV but not in open-ended:", (~sav["lfdn_key"].isin(open_ended["lfdn_key"])).sum())

Open-ended rows: 8562
Unique lfdn values: 8562
Duplicate lfdn rows: 0

SAV rows: 8562
SAV unique lfdn values: 8562
lfdn values in open-ended but not in SAV: 0
lfdn values in SAV but not in open-ended: 0


## 4. Which fields are text fields?

There is no single text column yet. The open-ended CSV has many text columns. For the later OpenCodeBook workflow, we will probably reshape them into one long column named `response_text`.

## Major Open-Ended Fields: Survey Wording

Based on the GLES questionnaire documentation for ZA10101:

- `pre020s`: pre-election open response for the most important political problem facing Germany today.
- `pre022s`: pre-election open response for the second most important political problem facing Germany today.
- `pos021s`: post-election open response for the most important political problem facing Germany today.
- `pos023s`: post-election open response for the second most important political problem facing Germany today.

The questionnaire asks respondents to name only one problem for each field. These fields should therefore be interpreted primarily as political problem / issue-salience responses, not as stance statements.

Related follow-up variables in the `.sav` ask which party is best able to handle the named problem:

- `pre021a` / `pre021b`: party best able to handle `pre020s`.
- `pre023a` / `pre023b`: party best able to handle `pre022s`.
- `pos022a` / `pos022b`: party best able to handle `pos021s`.
- `pos024a` / `pos024b`: party best able to handle `pos023s`.

Design implication: the GLES pilot should focus on issue domain, specificity, framing, and ambiguity. Stance should only be coded when it is explicit or strongly inferable.

Source: GLES ZA10101 questionnaire documentation, `ZA10101_fb_en_v0-3_2025-03-10_1.pdf`.

In [29]:
text_cols = [col for col in open_ended.columns if col not in {"lfdn", "lfdn_key"}]

print("Number of text-like columns:", len(text_cols))
text_cols

Number of text-like columns: 20


['pre020s',
 'pre022s',
 'pre041xs',
 'pre060s',
 'pre062s',
 'pre064s',
 'pre066os',
 'pre070xs',
 'pos021s',
 'pos023s',
 'pos026xs',
 'pos027xs',
 'pos035xs',
 'pos036xs',
 'pos037xs',
 'pos038xs',
 'pos039xs',
 'pos040os',
 'pos041xs',
 'pos042os']

In [ ]:
open_ended.head()

,lfdn,pre020s,pre022s,pre041xs,pre060s,pre062s,pre064s,pre066os,pre070xs,pos021s,...,pos027xs,pos035xs,pos036xs,pos037xs,pos038xs,pos039xs,pos040os,pos041xs,pos042os,lfdn_key
0,91,Migration,Innere Sicherheit,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,Migration,...,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,91
1,92,migranten,steuern,-97 trifft nicht zu,welt,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,migration,...,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,92
2,93,Wirtschaftliche Situation,Verteidigung,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,Wirtschaftskrise,...,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,93
3,94,Einigkeit,Bundestagsabgeordneten zu alt,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,Belange der deutschen Bevölkerung,...,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,94
4,95,Sozialbetrug,Einwanderung,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,Geld abfuss,...,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,-97 trifft nicht zu,95


## 5. Clean structural missing values

These strings are not substantive answers. We temporarily treat them as missing for inspection only. We are not editing the raw data.

In [36]:
missing_tokens = {
    "",
    "-97 trifft nicht zu",
    "-95 nicht teilgenommen",
    "-93 Interview abgebrochen",
    "-98 weiss nicht",
    "-98 weiß nicht",
    "-99 keine Angabe",
}

missing_tokens_folded = {token.casefold() for token in missing_tokens}

def clean_response(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if text.casefold() in missing_tokens_folded:
        return pd.NA
    return text

raw_values = pd.concat(
    [open_ended[col].fillna("").astype(str).str.strip() for col in text_cols],
    ignore_index=True,
)

raw_values.value_counts().head(15)

-97 trifft nicht zu          124302
-95 nicht teilgenommen         7492
Wirtschaft                     4131
Migration                      3682
-98 weiss nicht                2042
-99 keine Angabe                832
Inflation                       515
-93 Interview abgebrochen       494
Zuwanderung                     436
Klimawandel                     410
Sicherheit                      392
Asylpolitik                     343
Rente                           325
Bildung                         302
Einwanderung                    298
Name: count, dtype: int64

## 6. Missingness by text column

This tells us which columns are core response fields and which are sparse follow-up or `other specify` fields.

In [49]:
profile_rows = []

for col in text_cols:
    cleaned = open_ended[col].map(clean_response)
    substantive = cleaned.dropna()
    words = substantive.map(lambda text: len(re.findall(r"\w+", text)))
    chars = substantive.map(len)

    profile_rows.append(
        {
            "column": col,
            "substantive": int(substantive.size),
            "missing_or_non_substantive": int(cleaned.isna().sum()),
            "pct_substantive": round(100 * substantive.size / len(open_ended), 1),
            "unique_substantive": int(substantive.nunique()),
            "median_words": words.median() if substantive.size else pd.NA,
            "p90_words": words.quantile(0.9) if substantive.size else pd.NA,
            "median_chars": chars.median() if substantive.size else pd.NA,
            "p90_chars": chars.quantile(0.9) if substantive.size else pd.NA,
            "one_word": int((words <= 1).sum()) if substantive.size else 0,
            "two_words_or_less": int((words <= 2).sum()) if substantive.size else 0,
        }
    )

column_profile = pd.DataFrame(profile_rows)
column_profile

,column,substantive,missing_or_non_substantive,pct_substantive,unique_substantive,median_words,p90_words,median_chars,p90_chars,one_word,two_words_or_less
0,pre020s,8327,235,97.3,2879,1.0,4.0,11.0,32.0,5650,6754
1,pre022s,8270,292,96.6,2790,1.0,4.0,11.0,29.0,5931,6933
2,pre041xs,61,8501,0.7,38,2.0,8.0,12.0,50.0,19,45
3,pre060s,448,8114,5.2,213,1.0,3.0,4.5,18.0,246,368
4,pre062s,2275,6287,26.6,870,2.0,2.0,14.0,22.0,893,2080
5,pre064s,383,8179,4.5,247,1.0,3.0,8.0,20.0,222,339
6,pre066os,103,8459,1.2,68,1.0,2.0,6.0,17.8,73,96
7,pre070xs,29,8533,0.3,18,1.0,5.0,5.0,35.2,16,23
8,pos021s,7701,861,89.9,2390,1.0,3.0,10.0,28.0,5594,6489
9,pos023s,7653,909,89.4,2218,1.0,3.0,10.0,25.0,5812,6714


## 7. Reshape to long format for inspection

This gives one row per substantive text answer. This is probably the shape we want for a pilot sample later.

In [50]:
long_parts = []

for col in text_cols:
    part = open_ended[["lfdn", "lfdn_key"]].copy()
    part["source_var"] = col
    part["response_text"] = open_ended[col].map(clean_response)
    long_parts.append(part)

responses_long = pd.concat(long_parts, ignore_index=True).dropna(subset=["response_text"])
responses_long["response_norm"] = (
    responses_long["response_text"]
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
responses_long["n_words"] = responses_long["response_text"].map(lambda text: len(re.findall(r"\w+", text)))
responses_long["n_chars"] = responses_long["response_text"].map(len)

print("Substantive response cells:", len(responses_long))
print("Respondents with at least one substantive answer:", responses_long["lfdn_key"].nunique())
print("Respondents with no substantive answer:", open_ended.loc[~open_ended["lfdn_key"].isin(responses_long["lfdn_key"]), "lfdn_key"].nunique())

responses_long.head(20)

Substantive response cells: 36077
Respondents with at least one substantive answer: 8475
Respondents with no substantive answer: 87


,lfdn,lfdn_key,source_var,response_text,response_norm,n_words,n_chars
0,91,91,pre020s,Migration,migration,1,9
1,92,92,pre020s,migranten,migranten,1,9
2,93,93,pre020s,Wirtschaftliche Situation,wirtschaftliche situation,2,25
3,94,94,pre020s,Einigkeit,einigkeit,1,9
4,95,95,pre020s,Sozialbetrug,sozialbetrug,1,12
5,96,96,pre020s,Wirtschaft,wirtschaft,1,10
6,97,97,pre020s,Flüchtlinge,flüchtlinge,1,11
7,98,98,pre020s,Demokratiegefährdung durch Radikale,demokratiegefährdung durch radikale,3,35
8,99,99,pre020s,Flüchtlingskrise,flüchtlingskrise,1,16
9,100,100,pre020s,Kinderbetreuung,kinderbetreuung,1,15


## 8. Duplicates and very short answers

Repeated answers are expected here because many responses are issue labels like `Wirtschaft` or `Migration`.

In [51]:
print("Unique normalized answer strings:", responses_long["response_norm"].nunique())
print("Extra duplicate occurrences across all response cells:", len(responses_long) - responses_long["response_norm"].nunique())

print("\nWord count summary:")
display(responses_long["n_words"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

print("\nTop repeated substantive answers:")
display(responses_long["response_norm"].value_counts().head(25))

Unique normalized answer strings: 9149
Extra duplicate occurrences across all response cells: 26928

Word count summary:


count    36077.000000
mean         1.852427
std          2.351631
min          0.000000
10%          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
90%          3.000000
95%          5.000000
99%         12.000000
max         44.000000
Name: n_words, dtype: float64


Top repeated substantive answers:


response_norm
wirtschaft               4343
migration                3847
inflation                 547
zuwanderung               468
klimawandel               421
sicherheit                409
asylpolitik               354
rente                     350
bildung                   316
einwanderung              313
ausländer                 299
flüchtlinge               270
verteidigung              266
klima                     256
asyl                      242
infrastruktur             232
wirtschaftliche lage      214
steuern                   200
krieg                     196
soziale gerechtigkeit     184
klimaschutz               175
migrationspolitik         172
wirtschaftskrise          170
finanzen                  170
innere sicherheit         168
Name: count, dtype: int64

In [52]:
responses_long.loc[responses_long["n_words"] <= 1, ["lfdn", "source_var", "response_text", "n_words"]].head(30)

,lfdn,source_var,response_text,n_words
0,91,pre020s,Migration,1
1,92,pre020s,migranten,1
3,94,pre020s,Einigkeit,1
4,95,pre020s,Sozialbetrug,1
5,96,pre020s,Wirtschaft,1
6,97,pre020s,Flüchtlinge,1
8,99,pre020s,Flüchtlingskrise,1
9,100,pre020s,Kinderbetreuung,1
11,102,pre020s,Rente,1
13,104,pre020s,Konjunktur,1


In [53]:
responses_long.loc[responses_long["n_words"] >= 8, ["lfdn", "source_var", "response_text", "n_words"]].head(30)

,lfdn,source_var,response_text,n_words
61,155,pre020s,Endlich wieder Politik für die Bürger zu mache...,13
70,164,pre020s,nur Versager in den Ämtern eingesetzt nach Quo...,12
72,166,pre020s,Sozialstaat der nicht mehr Fordert sondern Soz...,9
133,230,pre020s,Privatbürger werden zu hohen Ausgaben gezwunge...,19
137,234,pre020s,Die gemäßigten Parteien bewegen sich alle star...,23
174,275,pre020s,Alles wichtig. Aber es sollte das tägliche Leb...,28
250,355,pre020s,Das sie zu wenig für uns deutsche machen,8
259,365,pre020s,"Kriminalität, Angst vor dem Weg zur Arbeit (we...",17
262,368,pre020s,Politik hat nichts mit den Wünschen des Volkes...,10
285,392,pre020s,"keine Kontinuität in der Politik, Entscheidung...",17


## 9. Metadata fields for stratifying a pilot sample

The open-ended CSV has only `lfdn` and text fields. The `.sav` has respondent metadata. Here we inspect a few candidate fields after joining by `lfdn`.

In [60]:
candidate_metadata = [
    "lfdn_key",
    "lfdn",
    "sample",
    "wave",
    "pre001",   # gender
    "pre002a",  # year of birth
    "pre003",   # school education
    "ostwest",
    "ostwest2",
    "bula",
    "sup004",   # federal state
    "sup057",   # left-right self-placement
    "pre009",   # vote participation intention
    "pre012ba", # second vote intention, version A
    "pre012bb", # second vote intention, version B
    "pre072a",  # party ID, version A
    "pre072b",  # party ID, version B
    "pre073",   # party ID strength
    "pos010ba", # second vote decision, version A
    "pos010bb", # second vote decision, version B
    "sup072",   # household language, German
    "sup073",   # household language, other
    "sup082",   # household net income
    "w_pre_ipf",
    "w_pos_ipf",
]

available_metadata = [col for col in candidate_metadata if col in sav.columns]
missing_metadata = [col for col in candidate_metadata if col not in sav.columns]

print("Available:", available_metadata)
print("Missing:", missing_metadata)

Available: ['lfdn_key', 'lfdn', 'sample', 'wave', 'pre001', 'pre002a', 'pre003', 'ostwest', 'ostwest2', 'bula', 'sup004', 'sup057', 'pre009', 'pre012ba', 'pre012bb', 'pre072a', 'pre072b', 'pre073', 'pos010ba', 'pos010bb', 'sup072', 'sup073', 'sup082', 'w_pre_ipf', 'w_pos_ipf']
Missing: []


In [61]:
metadata = sav[available_metadata].copy()
responses_with_metadata = responses_long.merge(metadata, on="lfdn_key", how="left", validate="many_to_one", suffixes=("", "_sav"))

print("Long responses:", responses_long.shape)
print("Long responses with metadata:", responses_with_metadata.shape)
responses_with_metadata.head()

Long responses: (36077, 7)
Long responses with metadata: (36077, 31)


,lfdn,lfdn_key,source_var,response_text,response_norm,n_words,n_chars,lfdn_sav,sample,wave,...,pre072a,pre072b,pre073,pos010ba,pos010bb,sup072,sup073,sup082,w_pre_ipf,w_pos_ipf
0,91,91,pre020s,Migration,migration,1,9,91.0,GLES Rolling Cross-Section 2025,"Pre-Election, Supplementary Wave/Post-Election",...,CDU/CSU,CDU/CSU,fairly strongly,CDU/CSU,CDU/CSU,Not applicable,Not applicable,3.000 to less than 4.000 Euro,0.912530,0.887921
1,92,92,pre020s,migranten,migranten,1,9,92.0,GLES Rolling Cross-Section 2025,"Pre-Election, Supplementary Wave/Post-Election",...,AfD,AfD,very strongly,AfD,AfD,Not applicable,Not applicable,less than 500 euros,1.055157,1.077088
2,93,93,pre020s,Wirtschaftliche Situation,wirtschaftliche situation,2,25,93.0,GLES Rolling Cross-Section 2025,"Pre-Election, Supplementary Wave/Post-Election",...,CDU/CSU,CDU/CSU,fairly strongly,CDU/CSU,CDU/CSU,Not applicable,Not applicable,10.000 Euro or more,0.923433,0.894038
3,94,94,pre020s,Einigkeit,einigkeit,1,9,94.0,GLES Rolling Cross-Section 2025,"Pre-Election, Supplementary Wave/Post-Election",...,No party,No party,Not applicable,BSW,BSW,German,Not applicable,1.250 to less than 1.500 Euro,1.035925,1.025131
4,95,95,pre020s,Sozialbetrug,sozialbetrug,1,12,95.0,GLES Rolling Cross-Section 2025,"Pre-Election, Supplementary Wave/Post-Election",...,No party,No party,Not applicable,AfD,AfD,Not applicable,Not applicable,4.000 to less than 5.000 Euro,1.334453,1.370930


In [65]:
for col in ["wave", "pre001", "pre003", "ostwest", "bula", "sup057", "pre009", "pre072a", "pos010ba", "sup072"]:
    if col in responses_with_metadata.columns:
        print(f"\n--- {col} ---")
        display(responses_with_metadata.drop_duplicates("lfdn")[col].value_counts(dropna=False).head(15))


--- wave ---


wave
Pre-Election, Supplementary Wave/Post-Election    7776
Pre-Election and Supplementary Wave                424
only Pre-Election                                  171
Pre-Election and Post-Election                     104
Name: count, dtype: int64


--- pre001 ---


pre001
male      4420
female    4055
Name: count, dtype: int64


--- pre003 ---


...

pre072a
CDU/CSU        2041
No party       1874
SPD            1523
AfD            1123
GRUENE          765
FDP             367
DIE LINKE       327
BSW             292
Other party     128
No answer        33
Don't know        2
Name: count, dtype: int64


--- pos010ba ---


pos010ba
CDU/CSU             2029
SPD                 1334
AfD                 1287
GRUENE               750
DIE LINKE            627
Unit nonresponse     595
No answer            414
BSW                  389
FDP                  363
Not applicable       348
Other party          294
Invalid vote          29
Break-off             16
Name: count, dtype: int64


--- sup072 ---


sup072
Not applicable          6247
German                  1753
Unit nonresponse         275
a different language     165
Break-off                 33
No answer                  2
Name: count, dtype: int64

## 10. First interpretation

Got the below information after running the notebook:

- One row in the raw open-ended CSV represents one respondent.
- There are many text columns, so the pilot probably needs a long `response_text` table.
- The main populated text fields appear to be `pre020s`, `pre022s`, `pos021s`, and `pos023s`.
- The language is German.
- Many answers are very short topic labels, so the coding task may be closer to issue/stance/topic interpretation than long-text annotation.
- Useful pilot strata likely come from the `.sav` metadata joined by `lfdn`.